In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# وضع الاختبار المحلي في Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو ما هو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
استخدم وضع الاختبار المحلي (المتاح مع الإصدار `qiskit-ibm-runtime` v0.22.0 أو أحدث) لاختبار البرامج قبل ضبطها وإرسالها إلى أجهزة الكم الحقيقية. بعد التحقق من برنامجك باستخدام وضع الاختبار المحلي، كل ما تحتاج إلى تغييره هو اسم الخلفية لتشغيله على وحدة المعالجة الكمومية (QPU).

لاستخدام وضع الاختبار المحلي، حدد إحدى الخلفيات الوهمية من ``qiskit_ibm_runtime.fake_provider`` أو حدد خلفية Qiskit Aer عند إنشاء primitive أو جلسة في Qiskit Runtime.

- **الخلفيات الوهمية**: تحاكي [الخلفيات الوهمية](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) في ``qiskit_ibm_runtime.fake_provider`` سلوكيات وحدات المعالجة الكمومية من IBM&reg; باستخدام لقطات QPU. تحتوي هذه اللقطات على معلومات مهمة حول وحدة المعالجة الكمومية، مثل خريطة التوصيل والبوابات الأساسية وخصائص الكيوبت، وهي مفيدة لاختبار المحوّل وإجراء محاكاة QPU مع الضوضاء. يُطبَّق نموذج الضوضاء من اللقطة تلقائيًا أثناء المحاكاة.
- **محاكي Aer**: توفر المحاكيات من [Qiskit Aer](/guides/simulate-with-qiskit-aer) محاكاة عالية الأداء تتعامل مع دوائر أكبر و[نماذج ضوضاء مخصصة](/guides/build-noise-models). تتوفر قائمة بخيارات طرق المحاكاة عند استخدام `AerSimulator` في وضع الاختبار المحلي. اطلع على [مثال وضع محاكاة Clifford](#clifford-sim) الذي يوضح كيفية محاكاة دوائر Clifford ذات عدد كبير من الكيوبتات بكفاءة.

    <details>
    <summary>**قائمة طرق المحاكاة المتاحة من Qiskit Aer**</summary>

    راجع توثيق [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) للمزيد من المعلومات.

    * ``"automatic"``: طريقة المحاكاة الافتراضية. تختار طريقة المحاكاة تلقائيًا بناءً على الدائرة ونموذج الضوضاء.

    * ``"statevector"``: محاكاة كثيفة لمتجه الحالة يمكنها أخذ عينات من نتائج القياس من الدوائر *المثالية* مع جميع القياسات في نهاية الدائرة. في حالة المحاكاة مع الضوضاء، تأخذ كل لقطة عينة من دائرة عشوائية مع ضوضاء من نموذج الضوضاء.

    * ``"density_matrix"``: محاكاة مصفوفة الكثافة التي يمكنها أخذ عينات من نتائج القياس من الدوائر *المضوضاة* مع جميع القياسات في نهاية الدائرة.

    * ``"stabilizer"``: محاكي فعّال لحالة Clifford المستقرة يمكنه محاكاة دوائر Clifford مع الضوضاء إذا كانت جميع الأخطاء في نموذج الضوضاء هي أيضًا أخطاء Clifford.

    * ``"extended_stabilizer"``: محاكي تقريبي لدوائر Clifford + T يعتمد على تحليل الحالة إلى حالة مستقرة مرتبة. يزداد عدد المصطلحات مع عدد البوابات غير المنتمية لـ Clifford (T).

    * ``"matrix_product_state"``: محاكي متجه الحالة بالشبكة التنسورية يستخدم تمثيل Matrix Product State (MPS) للحالة. يمكن تنفيذ ذلك مع أو بدون تقليص أبعاد ربط MPS، وفقًا لخيارات المحاكي. السلوك الافتراضي هو عدم التقليص.

    * ``"unitary"``: محاكاة كثيفة لمصفوفة الوحدة لدائرة مثالية. تحاكي المصفوفة الوحدوية للدائرة ذاتها بدلاً من تطور الحالة الكمومية الابتدائية. تدعم هذه الطريقة البوابات فقط ولا تدعم القياس أو إعادة الضبط أو الضوضاء.

    * ``"superop"``: محاكاة كثيفة لمصفوفة superoperator لدائرة مثالية أو مضوضاة. تحاكي مصفوفة superoperator للدائرة ذاتها بدلاً من تطور الحالة الكمومية الابتدائية. تدعم هذه الطريقة محاكاة البوابات وإعادة الضبط، سواء كانت مثالية أو مضوضاة، لكنها لا تدعم القياسات.

    * ``"tensor_network"``: محاكاة تعتمد على الشبكة التنسورية وتدعم كلًا من متجه الحالة ومصفوفة الكثافة. حاليًا، هذه الطريقة متاحة فقط لوحدات GPU وتعمل بتسريع باستخدام واجهات برمجة تطبيقات cuQuantum `cuTensorNet`.
    </details>

> **Note:** - يمكنك تحديد جميع خيارات Qiskit Runtime في وضع الاختبار المحلي. غير أن جميع الخيارات باستثناء shots تُتجاهل عند التشغيل على محاكي محلي.
> - يُنصح بتثبيت Qiskit Aer قبل استخدام الخلفيات الوهمية أو محاكيات Aer عبر تشغيل `pip install qiskit-aer`. تستخدم الخلفيات الوهمية محاكيات Aer في الخلفية إذا كانت متاحة، للاستفادة من أدائها.


## مثال على الخلفيات الوهمية

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## أمثلة على AerSimulator
مثال مع الجلسات، بدون ضوضاء:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

لمحاكاة مع الضوضاء، حدد وحدة معالجة كمومية (QPU) وأرسلها إلى Aer. يبني Aer نموذج ضوضاء بناءً على بيانات المعايرة من تلك الوحدة، ويهيئ خلفية Aer بهذا النموذج. إذا كنت تفضل ذلك، يمكنك [بناء نموذج ضوضاء مخصص](/guides/build-noise-models).

> **Caution:** قد تتأثر وحدة المعالجة الكمومية بأنواع مختلفة من الضوضاء. نموذج ضوضاء Qiskit Aer المستخدم هنا يحاكي بعضها فقط، وبالتالي من المرجح أن يكون أقل حدة من الضوضاء الموجودة في وحدة المعالجة الكمومية الحقيقية.
> 
> للاطلاع على تفاصيل الأخطاء المُضمَّنة عند تهيئة نموذج ضوضاء من وحدة QPU، راجع مرجع Aer API لـ [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

مثال مع الضوضاء:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### محاكاة Clifford
نظرًا لإمكانية محاكاة دوائر Clifford بكفاءة مع نتائج قابلة للتحقق، تُعد محاكاة Clifford أداة بالغة الأهمية. للاطلاع على مثال تفصيلي، راجع [المحاكاة الفعّالة لدوائر الاستقرار مع Qiskit Aer primitives](/guides/simulate-stabilizer-circuits).

مثال:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## الخطوات التالية
> **Tip:** - راجع [أمثلة تفصيلية على primitives.](/guides/primitives-examples)
>     - اقرأ [الانتقال إلى V2 primitives](/guides/v2-primitives).
>     - تدرّب على استخدام primitives من خلال [درس دالة التكلفة](/learning/courses/variational-algorithm-design/cost-functions) في IBM Quantum Learning.
>     - تعلّم كيفية التحويل البرمجي محليًا في قسم [التحويل البرمجي](/guides/transpile).
>     - جرّب تطبيق [مقارنة إعدادات المحوّل](/guides/circuit-transpilation-settings#compare-transpiler-settings).